In [ ]:
%pip install --quiet python-dotenv pydantic-ai whisperx 'nemo_toolkit[asr]'

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/03-audio-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Load API keys: Colab secrets or local .env
try:
    from google.colab import userdata
    os.environ.setdefault("GOOGLE_API_KEY", userdata.get("GOOGLE_API_KEY"))
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Audio

What can you do with audio? Turn it into text, then do text things. Split it by speaker.

## Transcribe

Once upon a time OpenAI released an *open model* named Whisper. It's actually great! There are newer models out there - like [Parakeet](https://parakeettdt.com/), which is super quick - but Whisper is very easy to use so everyone runs it.


In [ ]:
from IPython.display import Audio
Audio(filename="data/rDXubdQdJYs.mp3")


There are different versions of the model - like tiny, base, large - but **turbo** is the best combination of speed and accuracy. Below we're using **WhisperX** which is a feature-packed tool built on top of Whisper.


**`audio/whisperx.py`** — WhisperX segment-level aligned transcription


In [ ]:
from pathlib import Path
import torch
import whisperx

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL = "large-v3"
LANGUAGE = "en"

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

model = whisperx.load_model(MODEL, device, compute_type=compute_type)
audio = whisperx.load_audio(str(AUDIO))
result = model.transcribe(audio, batch_size=16)

model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=device)
result = whisperx.align(
    result["segments"], model_a, metadata, audio, device,
    return_char_alignments=False,
)

# Print out the entire transcript
print(" ".join(seg["text"].strip() for seg in result["segments"]))


/Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/lib/python3.12/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.8.0) is not compatible with

[NeMo W 2026-03-02 16:38:21 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


W0302 16:38:21.830000 13596 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.


No exporters were provided. This means that no telemetry data will be collected.


2026-03-02 16:38:25 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)


2026-03-02 16:38:25 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.4.0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


2026-03-02 16:38:33 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio


relative to what we're going to do with more border patrol and more asylum officers. President Trump? I really don't know what he said at the end of that sentence. I don't think he knows what he said either. The only person in this stage is a convicted felon is the man I'm looking at right now. But when he talks about a convicted felon, his son is a convicted felon. What are you talking about? You have the morals of an alley cat. My son was not a loser, was not a sucker. You're the sucker. You're the loser. He's blaming inflation, and he's right. It's been very bad. He caused the inflation. Excuse me, with dealing with everything we have to do with... Look, if we finally beat Medicare... Thank you, President Biden. President Trump? Well, I took two tests, cognitive tests. I aced them. Go through the first five questions. He couldn't do it. This guy's three years younger and a lot less competent. I think that just look at the record. Look at what I've done.


You can also view the segments with timestamps and aligned text


In [ ]:
import pandas as pd

df = pd.DataFrame(result["segments"])
df


,start,end,text,words
0,2.461,7.990,relative to what we're going to do with more ...,"[{'word': 'relative', 'start': 2.461, 'end': 2..."
1,8.010,9.091,President Trump?,"[{'word': 'President', 'start': 8.01, 'end': 8..."
2,9.111,11.215,I really don't know what he said at the end of...,"[{'word': 'I', 'start': 9.111, 'end': 9.151, '..."
3,11.555,12.897,I don't think he knows what he said either.,"[{'word': 'I', 'start': 11.555, 'end': 11.575,..."
4,12.937,16.904,The only person in this stage is a convicted f...,"[{'word': 'The', 'start': 12.937, 'end': 13.23..."
5,17.324,20.629,"But when he talks about a convicted felon, his...","[{'word': 'But', 'start': 17.324, 'end': 17.42..."
6,20.750,22.292,What are you talking about?,"[{'word': 'What', 'start': 20.75, 'end': 20.87..."
7,22.913,25.197,You have the morals of an alley cat.,"[{'word': 'You', 'start': 22.913, 'end': 23.35..."
8,25.217,27.200,"My son was not a loser, was not a sucker.","[{'word': 'My', 'start': 25.217, 'end': 25.357..."
9,27.481,28.182,You're the sucker.,"[{'word': 'You're', 'start': 27.481, 'end': 27..."


## Transcribe

Here's Parakeet! If you're on a mac, use [parakeet-mlx](https://github.com/senstella/parakeet-mlx). I'm specifically using the *most complicated possible version of Parakeet* because it's the best.


**`audio/parakeet.py`** — Parakeet transcription via NeMo (NVIDIA Parakeet TDT with timestamps)


In [ ]:
from pathlib import Path
import nemo.collections.asr as nemo_asr

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL = "nvidia/parakeet-tdt-0.6b-v2"

model = nemo_asr.models.ASRModel.from_pretrained(model_name=MODEL)
output = model.transcribe([str(AUDIO)], timestamps=True, channel_selector=0)

for stamp in output[0].timestamp["segment"]:
    start = f"{int(stamp['start']//60):02d}:{stamp['start']%60:05.2f}"
    end = f"{int(stamp['end']//60):02d}:{stamp['end']%60:05.2f}"
    print(f"[{start} - {end}] {stamp['segment']}")


[NeMo I 2026-03-02 16:38:59 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-02 16:38:59 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    


[NeMo W 2026-03-02 16:38:59 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_lhotse: true
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 16
    shuffle: false
    max_duration: 40.0
    min_duration: 0.1
    num_workers: 2
    pin_memory: true
    text_field: answer
    


[NeMo I 2026-03-02 16:39:01 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo I 2026-03-02 16:39:01 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


...

[NeMo W 2026-03-02 16:39:02 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


Transcribing: 0it [00:00, ?it/s]

Transcribing: 1it [00:02,  3.00s/it]

Transcribing: 1it [00:02,  3.00s/it]

[00:02.48 - 00:07.52] Relative to what we can do with more Border Patrol and more asylum officers.
[00:07.84 - 00:08.48] President Trump?
[00:08.96 - 00:11.20] I really don't know what he said at the end of that sentence.
[00:11.44 - 00:12.80] I don't think he knows what he said either.
[00:13.12 - 00:16.96] The only person in this stage who's a convicted felon is the man I'm looking at right now.
[00:17.28 - 00:20.56] But when he talks about a convicted felon, his son is a convicted felon.
[00:21.20 - 00:22.56] What are you talking about?
[00:23.28 - 00:24.80] You have the morals of an alley cat.
[00:25.12 - 00:27.20] My son was not a loser, was not a sucker.
[00:27.44 - 00:28.16] You're the sucker.
[00:28.32 - 00:29.28] You're the loser.
[00:29.60 - 00:31.92] He's blaming inflation, and he's right.
[00:32.00 - 00:33.12] It's been very bad.
[00:33.44 - 00:34.64] He caused the inflation.
[00:35.04 - 00:39.20] Excuse me, with dealing with everything we have to do with.
[00:40.96 - 00:46

## Transcribe and diarize (local)

Along with transcribing, WhisperX transcribes the audio *and identifies who said what*. Free, runs locally (on your own computer), your audio never leaves your machine.

**Setup:** Diarization requires a free Hugging Face token (`HF_TOKEN`) and accepting the model licenses at [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) and [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1).


**`audio/whisperx-diarize.py`** — Full WhisperX pipeline: transcribe, align, diarize (who said what)


In [ ]:
from pathlib import Path
import os
from collections import defaultdict
import torch, whisperx
from whisperx.diarize import DiarizationPipeline

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL, LANGUAGE = "large-v3", "en"
HF_TOKEN = os.environ["HF_TOKEN"]
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

# Step 1: Transcribe
model = whisperx.load_model(MODEL, device, compute_type=compute_type)
audio = whisperx.load_audio(str(AUDIO))
result = model.transcribe(audio, batch_size=16)
# Step 2: Align
model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
# Step 3: Diarize
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
result = whisperx.assign_word_speakers(diarize_model(audio), result)

# Print speaker-labeled segments
current_speaker = None
for seg in result["segments"]:
    speaker = seg.get("speaker", "UNKNOWN")
    if speaker != current_speaker:
        print(f"\n--- {speaker} ---")
        current_speaker = speaker
    start = f"{int(seg['start']//60):02d}:{seg['start']%60:05.2f}"
    end = f"{int(seg['end']//60):02d}:{seg['end']%60:05.2f}"
    print(f"  [{start} - {end}] {seg['text'].strip()}")

# Speaker time summary
speaker_time = defaultdict(float)
for seg in result["segments"]:
    speaker_time[seg.get("speaker", "UNKNOWN")] += seg["end"] - seg["start"]
total = sum(speaker_time.values())
print(f"\n{'Speaker':<12} {'Time':>8} {'%':>6}")
for spk in sorted(speaker_time):
    t = speaker_time[spk]
    print(f"  {spk:<10} {t/60:>6.1f}m {t/total*100:>5.1f}%")


2026-03-02 16:39:08 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)


2026-03-02 16:39:08 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.4.0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


2026-03-02 16:39:15 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio


2026-03-02 16:39:40 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1



--- SPEAKER_01 ---
  [00:02.46 - 00:07.99] relative to what we're going to do with more border patrol and more asylum officers.

--- SPEAKER_00 ---
  [00:08.01 - 00:09.09] President Trump?
  [00:09.11 - 00:11.21] I really don't know what he said at the end of that sentence.
  [00:11.55 - 00:12.90] I don't think he knows what he said either.

--- SPEAKER_01 ---
  [00:12.94 - 00:16.90] The only person in this stage is a convicted felon is the man I'm looking at right now.

--- SPEAKER_00 ---
  [00:17.32 - 00:20.63] But when he talks about a convicted felon, his son is a convicted felon.

--- SPEAKER_01 ---
  [00:20.75 - 00:22.29] What are you talking about?
  [00:22.91 - 00:25.20] You have the morals of an alley cat.
  [00:25.22 - 00:27.20] My son was not a loser, was not a sucker.
  [00:27.48 - 00:28.18] You're the sucker.
  [00:28.34 - 00:29.02] You're the loser.

--- SPEAKER_00 ---
  [00:29.70 - 00:32.07] He's blaming inflation, and he's right.
  [00:32.09 - 00:32.95] It's been very 

In [ ]:
import pandas as pd
df = pd.DataFrame(result["segments"])
df


,start,end,text,words,speaker
0,2.461,7.990,relative to what we're going to do with more ...,"[{'word': 'relative', 'start': 2.461, 'end': 2...",SPEAKER_01
1,8.010,9.091,President Trump?,"[{'word': 'President', 'start': 8.01, 'end': 8...",SPEAKER_00
2,9.111,11.215,I really don't know what he said at the end of...,"[{'word': 'I', 'start': 9.111, 'end': 9.151, '...",SPEAKER_00
3,11.555,12.897,I don't think he knows what he said either.,"[{'word': 'I', 'start': 11.555, 'end': 11.575,...",SPEAKER_00
4,12.937,16.904,The only person in this stage is a convicted f...,"[{'word': 'The', 'start': 12.937, 'end': 13.23...",SPEAKER_01
5,17.324,20.629,"But when he talks about a convicted felon, his...","[{'word': 'But', 'start': 17.324, 'end': 17.42...",SPEAKER_00
6,20.750,22.292,What are you talking about?,"[{'word': 'What', 'start': 20.75, 'end': 20.87...",SPEAKER_01
7,22.913,25.197,You have the morals of an alley cat.,"[{'word': 'You', 'start': 22.913, 'end': 23.35...",SPEAKER_01
8,25.217,27.200,"My son was not a loser, was not a sucker.","[{'word': 'My', 'start': 25.217, 'end': 25.357...",SPEAKER_01
9,27.481,28.182,You're the sucker.,"[{'word': 'You're', 'start': 27.481, 'end': 27...",SPEAKER_01


"Speaker 1 said X at 0:42, Speaker 2 said Y at 1:15." Now you have a searchable, speaker-labeled transcript.

If you'd prefer to not write code (lol), try [MacWhisper](https://goodsnooze.gumroad.com/l/macwhisper), [Handy](https://handy.computer/), [VoiceInk](https://tryvoiceink.com/), [Buzz](https://buzzcaptions.com/).

## Structured transcription (cloud)

Sometimes your computer is slow, or you don't care about privacy or cost, and you just want something to get *done*. Here's the same audio, but Gemini returns structured data — each utterance as a typed object with speaker, timestamps, and text. Same Pydantic AI pattern as the image notebooks.


**`audio/gemini-diarize.py`** — Structured transcription with speaker labels via Gemini (cloud alternative to WhisperX)


In [ ]:
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL = "google-gla:gemini-2.5-flash"

class Utterance(BaseModel):
    speaker: str = Field(description="Speaker label (e.g., Speaker 1, Speaker 2)")
    start: str = Field(description="Start timestamp MM:SS")
    end: str = Field(description="End timestamp MM:SS")
    text: str = Field(description="What was said")

agent = Agent(MODEL, output_type=list[Utterance])
result = agent.run_sync([
    "Transcribe this audio with speaker labels and timestamps for each utterance.",
    BinaryContent(data=AUDIO.read_bytes(), media_type="audio/mpeg"),
])
for u in result.output:
    print(f"[{u.start} - {u.end}] {u.speaker}: {u.text}")


[00:02 - 00:07] Speaker 1: relative to what we're going to do with more border patrol and more asylum officers.
[00:08 - 00:08] Speaker 2: President Trump.
[00:08 - 00:12] Speaker 3: I I really don't know what he said at the end of that sentence. I don't think he knows what he said either.
[00:13 - 00:16] Speaker 1: The only person at this stage is a convicted felon as the man I'm looking at right now.
[00:17 - 00:20] Speaker 3: But when he talks about a convicted felon, his son is a convicted felon.
[00:20 - 00:22] Speaker 1: What what are you talking about?
[00:22 - 00:24] Speaker 1: You you have the morals of an alley cat.
[00:25 - 00:29] Speaker 1: My son was not a loser, was not a sucker. You're the sucker. You're the loser.
[00:29 - 00:34] Speaker 3: He's blaming inflation, and he's right, it's been very bad. He caused the inflation.
[00:35 - 00:42] Speaker 1: excuse me, with um, dealing with everything we have to do with uh, look,
[00:43 - 00:46] Speaker 1: if we finally beat Me

Cloud tradeoff: faster, structured output built-in, but your audio goes to Google. Most newsrooms will use both local and cloud depending on the sensitivity of the material.
